## Demonstração do Controller de Usuário e Conta (Relacionamento)

Este notebook demonstra o funcionamento do controller de usuários-contas da API, cobrindo as seguintes operações de relacionamentos:

- Criar relação entre usuário e conta  (`POST /users_accounts/users/{user_id}/accounts/{account_id}`)
- Buscar contas por usuário (`GET /users_accounts/accounts/{user_id}/users`)
- Buscar usuários por conta (`GET /users_accounts/users/{account_id}/accounts`)
- Deletar relação entre usuário e conta (`DELETE /users_accounts/users/{user_id}/accounts/{account_id}`)


## Setup do Teste com FastAPI e TestClient

Import e configuração do FastAPI com o router `users_accounts`.

In [4]:
from fastapi.testclient import TestClient
from fastapi import FastAPI

from user.controller import users_router
from accounts.controller import accounts_router
from user_account.controller import user_account_router

app = FastAPI()
app.include_router(users_router)
app.include_router(accounts_router)
app.include_router(user_account_router)

client = TestClient(app)

## Criar Usuário e Conta de Teste

**Endpoint:** 
`POST /users/create`  
`POST /accounts/create` 
**Descrição:** Cria um novo usuário e conta com os dados fornecidos no payload.

In [5]:
user_payload = {
    "first_name": "Stuart",
    "last_name": "Price",
    "cpf": "961.456.040-18",
    "email": "stuprice@email.com",
    "password": "Strong@Password1976",
    "manual_balance": 100.0
}

response = client.post("/users/create", json=user_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_user = response.json()
user_id = created_user["id"]

account_payload = {
   "name": "Conta Corrente de Stu",
   "type": "Corrente",
   "account_number": "SP20091125254622633269",
   "balance": 50000,
}

response = client.post("/accounts/create", json=account_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_account = response.json()
account_id = created_account["id"]

2025-06-15 14:02:35,219 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:02:35,222 INFO sqlalchemy.engine.Engine INSERT INTO users (id, first_name, last_name, email, cpf, password, manual_balance, created_at, updated_at, deleted_at) VALUES (%(id)s, %(first_name)s, %(last_name)s, %(email)s, %(cpf)s, %(password)s, %(manual_balance)s, %(created_at)s, %(updated_at)s, %(deleted_at)s)
2025-06-15 14:02:35,223 INFO sqlalchemy.engine.Engine [cached since 84.37s ago] {'id': '0684efcab38b7d47800088fe56c3027a', 'first_name': 'Stuart', 'last_name': 'Price', 'email': 'stuprice@email.com', 'cpf': '96145604018', 'password': '$2b$12$lyPN95DfBQ25aD7RGSvf4e51leGuos84MQGvmB.HTgBe4NFJAuvPG', 'manual_balance': 100.0, 'created_at': datetime.datetime(2025, 6, 15, 14, 1, 10, 281016), 'updated_at': datetime.datetime(2025, 6, 15, 14, 1, 10, 282071), 'deleted_at': None}
2025-06-15 14:02:35,225 INFO sqlalchemy.engine.Engine COMMIT
2025-06-15 14:02:35,233 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2

## Criar relação entre usuário e conta

**Endpoint:** 
`POST /users/{user_id}/accounts/{account_id}` 
**Descrição:** Cria relação entre usuário e conta com os dados fornecidos na requisição.

In [6]:
response = client.post(f"/users_accounts/users/{user_id}/accounts/{account_id}")
print("Status:", response.status_code)
print("Resposta:", response.json())

Status: 404
Resposta: {'detail': 'Not Found'}


## Buscar conta(s) por usuário específico através do id

**Endpoint:** `GET /users_accounts/accounts/{user_id}/users`  
**Descrição:** Retorna todas as contas relacionados ao usuário posto no endpoint.

In [7]:
response = client.get(f"/accounts/{user_id}/users")
print("Status:", response.status_code)
print("Usuário(s):", response.json())

2025-06-15 14:04:12,896 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:04:12,909 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.first_name AS users_first_name, users.last_name AS users_last_name, users.email AS users_email, users.cpf AS users_cpf, users.password AS users_password, users.manual_balance AS users_manual_balance, users.created_at AS users_created_at, users.updated_at AS users_updated_at, users.deleted_at AS users_deleted_at 
FROM users 
WHERE users.id = %(id_1)s AND users.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-15 14:04:12,911 INFO sqlalchemy.engine.Engine [generated in 0.00229s] {'id_1': '0684efcab38b7d47800088fe56c3027a', 'param_1': 1}
2025-06-15 14:04:12,921 INFO sqlalchemy.engine.Engine SELECT accounts.id AS accounts_id, accounts.name AS accounts_name, accounts.type AS accounts_type, accounts.account_number AS accounts_account_number, accounts.balance AS accounts_balance, accounts.created_at AS accounts_created_at, accounts.update

## Buscar usuário(s) por conta específica através do id da conta

**Endpoint:** `GET /users/{account_id}/accounts`  
**Descrição:** Retorna todos os usuários relacionados a uma conta posta no endpoint.

In [9]:
response = client.get(f"/users_accounts/users/{account_id}/accounts")
print("Status:", response.status_code)
print("Conta(s):", response.json())

2025-06-15 14:05:48,089 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:05:48,094 INFO sqlalchemy.engine.Engine SELECT user_accounts.id AS user_accounts_id, user_accounts.user_id AS user_accounts_user_id, user_accounts.account_id AS user_accounts_account_id, user_accounts.created_at AS user_accounts_created_at 
FROM user_accounts 
WHERE user_accounts.account_id = %(account_id_1)s
2025-06-15 14:05:48,096 INFO sqlalchemy.engine.Engine [generated in 0.00116s] {'account_id_1': '0684efcab45e733a80004716bfe587b0'}
2025-06-15 14:05:48,099 INFO sqlalchemy.engine.Engine ROLLBACK
Status: 200
Conta(s): [{'id': '0684efd0-ceff-7aaa-8000-b39424f78bf7', 'user_id': '0684efca-b38b-7d47-8000-88fe56c3027a', 'account_id': '0684efca-b45e-733a-8000-4716bfe587b0', 'created_at': '2025-06-15T14:01:10'}]


## Deletar relação entre usuário e conta 

**Endpoint:** `DELETE /users/{user_id}/accounts/{account_id}`  
**Descrição:** Deleta permanentemente a relação entre usuário e conta do banco de dados (hard delete).

In [10]:
response = client.delete(f"/users_accounts/users/{user_id}/accounts/{account_id}")
print("Status:", response.status_code)
print("Forçando a Deleção no banco de dados:", response.json())

2025-06-15 14:06:17,940 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:06:17,945 INFO sqlalchemy.engine.Engine SELECT user_accounts.id AS user_accounts_id, user_accounts.user_id AS user_accounts_user_id, user_accounts.account_id AS user_accounts_account_id, user_accounts.created_at AS user_accounts_created_at 
FROM user_accounts 
WHERE user_accounts.user_id = %(user_id_1)s AND user_accounts.account_id = %(account_id_1)s 
 LIMIT %(param_1)s
2025-06-15 14:06:17,946 INFO sqlalchemy.engine.Engine [cached since 125s ago] {'user_id_1': '0684efcab38b7d47800088fe56c3027a', 'account_id_1': '0684efcab45e733a80004716bfe587b0', 'param_1': 1}
2025-06-15 14:06:17,952 INFO sqlalchemy.engine.Engine DELETE FROM user_accounts WHERE user_accounts.id = %(id)s AND user_accounts.user_id = %(user_id)s AND user_accounts.account_id = %(account_id)s
2025-06-15 14:06:17,954 INFO sqlalchemy.engine.Engine [generated in 0.00147s] {'id': '0684efd0ceff7aaa8000b39424f78bf7', 'user_id': '0684efcab38b7d478

JSONDecodeError: Expecting value: line 1 column 1 (char 0)